# MIS 433 Final Project: AI Investment Signals

For this project, we are looking at a few major AI-related companies and seeing if stock trends, volume, volatility, and news sentiment can help give a short-term investment signal.

The companies we are using are NVDA, MSFT, GOOGL, AMZN, AMD, and AVGO. Instead of trying to predict the exact stock price, we are focusing on whether the stock might move up or down over the next 7 trading days.


## 1. Environment Setup

This section sets up the notebook. It works in Colab, but it can also run locally in VS Code from the project folder.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

    possible_project_dirs = [
        Path('/content/MIS433-Final-Project'),
        Path('/content/drive/MyDrive/MIS_433_Project'),
        Path.cwd()
    ]

    PROJECT_DIR = None
    for folder in possible_project_dirs:
        if (folder / 'README.md').exists() or (folder / 'notebooks').exists():
            PROJECT_DIR = folder
            break

    if PROJECT_DIR is None:
        PROJECT_DIR = Path('/content/drive/MyDrive/MIS_433_Project')
        PROJECT_DIR.mkdir(parents=True, exist_ok=True)

    os.chdir(PROJECT_DIR)
    print(f"Running in Google Colab: {PROJECT_DIR}")
else:
    current_dir = Path.cwd().resolve()
    PROJECT_DIR = current_dir.parent if current_dir.name == 'notebooks' else current_dir
    os.chdir(PROJECT_DIR)
    print(f"Running locally in VS Code: {PROJECT_DIR}")

Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('data/external').mkdir(parents=True, exist_ok=True)
Path('outputs/charts').mkdir(parents=True, exist_ok=True)
Path('outputs/model_results').mkdir(parents=True, exist_ok=True)
Path('outputs/screenshots').mkdir(parents=True, exist_ok=True)


In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn requests google-generativeai


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid")


## 2. Stock Data Collection

Here we use `yfinance` to pull the historical stock data from Yahoo Finance for each company.


In [ ]:
tickers = ["NVDA", "MSFT", "GOOGL", "AMZN", "AMD", "AVGO"]
start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")


In [ ]:
stock_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    group_by="ticker"
)

stock_data.head()


In [ ]:
stock_data.to_csv("data/raw/stock_prices_raw.csv")


## 3. Data Cleaning

The stock data comes in a wide format, so this section cleans it up into one table that is easier to use.


In [ ]:
clean_data = []

for ticker in tickers:
    temp = stock_data[ticker].copy()
    temp["ticker"] = ticker
    temp = temp.reset_index()
    clean_data.append(temp)

stock_df = pd.concat(clean_data, ignore_index=True)
stock_df.columns = [col.lower().replace(" ", "_") for col in stock_df.columns]
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_prices_clean.csv", index=False)


## 4. Exploratory Data Analysis

This section looks at the data with a few basic charts and summary stats so we can compare the companies.


In [ ]:
plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = stock_df[stock_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["close"], label=ticker)

plt.title("Historical Closing Prices for AI Infrastructure Stocks")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
stock_df.info()
stock_df.describe()


## 5. Feature Engineering

Here we add new columns that can help with the model, like returns, moving averages, volatility, and volume change.


In [ ]:
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["daily_return"] = stock_df.groupby("ticker")["close"].pct_change()
stock_df["return_7d"] = stock_df.groupby("ticker")["close"].pct_change(7)
stock_df["return_30d"] = stock_df.groupby("ticker")["close"].pct_change(30)

stock_df["ma_7d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=7).mean())
stock_df["ma_30d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=30).mean())
stock_df["ma_90d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=90).mean())

stock_df["volatility_30d"] = stock_df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(window=30).std())
stock_df["volume_change"] = stock_df.groupby("ticker")["volume"].pct_change()

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_features.csv", index=False)


## 6. Stock Performance Comparisons

This section compares the stocks visually. The normalized chart helps because the companies all have different stock prices.


In [ ]:
normalized_df = stock_df.copy()
normalized_df["normalized_close"] = normalized_df.groupby("ticker")["close"].transform(lambda x: x / x.iloc[0] * 100)

plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = normalized_df[normalized_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["normalized_close"], label=ticker)

plt.title("Normalized Stock Performance Since 2020")
plt.xlabel("Date")
plt.ylabel("Indexed Price: 2020 = 100")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/charts/normalized_stock_performance.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
volatility_summary = stock_df.groupby("ticker")["daily_return"].std().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
volatility_summary.plot(kind="bar")
plt.title("Volatility Comparison by Stock")
plt.xlabel("Ticker")
plt.ylabel("Standard Deviation of Daily Returns")
plt.tight_layout()
plt.savefig("outputs/charts/volatility_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
summary = stock_df.groupby("ticker").agg(
    start_price=("close", "first"),
    end_price=("close", "last"),
    avg_daily_return=("daily_return", "mean"),
    volatility=("daily_return", "std"),
    avg_volume=("volume", "mean")
).reset_index()

summary["total_return"] = (summary["end_price"] / summary["start_price"]) - 1
summary = summary.sort_values("total_return", ascending=False)

summary


## 7. News Sentiment Data from Alpha Vantage

This section pulls recent news sentiment from Alpha Vantage. The sentiment score is positive for more positive news, negative for more negative news, and close to zero for neutral news.

The API key can come from Colab Secrets or from a local `.env` file. This keeps the key out of the notebook and out of GitHub.


In [ ]:
import os
import requests
import time

try:
    from google.colab import userdata
    alpha_key = userdata.get("ALPHA_VANTAGE_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    alpha_key = os.getenv("ALPHA_VANTAGE_API_KEY")

if not alpha_key or alpha_key == "your_alpha_vantage_api_key_here":
    raise ValueError("Alpha Vantage API key not found. Add it to Colab Secrets or your local .env file.")

all_sentiment = []

for ticker in tickers:
    print(f"Pulling sentiment for {ticker}...")

    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "apikey": alpha_key,
        "limit": 50
    }

    response = requests.get("https://www.alphavantage.co/query", params=params)
    data = response.json()

    if "feed" not in data:
        print(f"No feed returned for {ticker}: {data}")
        continue

    for article in data.get("feed", []):
        for ticker_info in article.get("ticker_sentiment", []):
            if ticker_info["ticker"] == ticker:
                all_sentiment.append({
                    "ticker": ticker,
                    "date": article["time_published"][:8],
                    "title": article["title"],
                    "source": article["source"],
                    "url": article["url"],
                    "ticker_sentiment_score": float(ticker_info["ticker_sentiment_score"]),
                    "ticker_sentiment_label": ticker_info["ticker_sentiment_label"],
                    "relevance_score": float(ticker_info["relevance_score"])
                })

    time.sleep(12)

sentiment_df = pd.DataFrame(all_sentiment)

if sentiment_df.empty:
    print("No new sentiment rows returned. The next cell will use saved daily sentiment if available.")

sentiment_df.head()


In [ ]:
daily_sentiment_path = Path("data/external/daily_sentiment_scores.csv")
using_saved_daily_sentiment = False

if sentiment_df.empty:
    if daily_sentiment_path.exists():
        print("Using saved daily sentiment scores.")
        daily_sentiment = pd.read_csv(daily_sentiment_path)
        daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"], errors="coerce")
        using_saved_daily_sentiment = True
    else:
        print("No sentiment data available. Creating empty sentiment columns.")
        daily_sentiment = pd.DataFrame(columns=[
            "ticker", "date", "avg_sentiment_score", "avg_relevance_score", "article_count"
        ])
else:
    sentiment_df["date"] = pd.to_datetime(sentiment_df["date"], errors="coerce")

    if sentiment_df["date"].isna().all() and daily_sentiment_path.exists():
        print("Raw sentiment dates were not usable, so using saved daily sentiment scores.")
        daily_sentiment = pd.read_csv(daily_sentiment_path)
        daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"], errors="coerce")
        using_saved_daily_sentiment = True
    else:
        daily_sentiment = sentiment_df.groupby(["ticker", "date"]).agg(
            avg_sentiment_score=("ticker_sentiment_score", "mean"),
            avg_relevance_score=("relevance_score", "mean"),
            article_count=("title", "count")
        ).reset_index()

daily_sentiment


In [ ]:
if not sentiment_df.empty and not using_saved_daily_sentiment:
    sentiment_df.to_csv("data/external/alpha_vantage_news_sentiment_raw.csv", index=False)
else:
    print("Skipping raw sentiment save because saved daily sentiment was used.")

daily_sentiment.to_csv("data/external/daily_sentiment_scores.csv", index=False)

daily_sentiment["ticker"].value_counts()


## 8. Combining Stock and Sentiment Data

Here we add the latest sentiment score for each company to the stock dataset so it can be used as another feature.


In [ ]:
latest_sentiment = daily_sentiment.sort_values("date").groupby("ticker").tail(1)

latest_sentiment = latest_sentiment[[
    "ticker",
    "avg_sentiment_score",
    "avg_relevance_score",
    "article_count"
]]

stock_df = pd.read_csv("data/processed/stock_features.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df = stock_df.merge(latest_sentiment, on="ticker", how="left")

stock_df["avg_sentiment_score"] = stock_df["avg_sentiment_score"].fillna(0)
stock_df["avg_relevance_score"] = stock_df["avg_relevance_score"].fillna(0)
stock_df["article_count"] = stock_df["article_count"].fillna(0)

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_features_with_sentiment.csv", index=False)


In [ ]:
sentiment_summary = latest_sentiment.sort_values("avg_sentiment_score", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(sentiment_summary["ticker"], sentiment_summary["avg_sentiment_score"])
plt.title("Latest Average News Sentiment by Company")
plt.xlabel("Ticker")
plt.ylabel("Average Sentiment Score")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.savefig("outputs/charts/latest_sentiment_by_company.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Prediction Target

This section creates the target column for the model. A `1` means the stock went up 7 trading days later, and a `0` means it did not.


In [ ]:
stock_df = pd.read_csv("data/processed/stock_features_with_sentiment.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["future_close_7d"] = stock_df.groupby("ticker")["close"].shift(-7)
stock_df["future_return_7d"] = (stock_df["future_close_7d"] / stock_df["close"]) - 1
stock_df["target_up_7d"] = (stock_df["future_return_7d"] > 0).where(
    stock_df["future_close_7d"].notna()
)
stock_df["target_up_7d"] = stock_df["target_up_7d"].astype("Int64")

feature_columns = [
    "daily_return", "return_7d", "return_30d", "ma_7d", "ma_30d",
    "volatility_30d", "volume_change", "avg_sentiment_score", "article_count"
]

training_df = stock_df.dropna(subset=feature_columns + ["target_up_7d"]).copy()
training_df["target_up_7d"] = training_df["target_up_7d"].astype(int)
latest_prediction_df = stock_df[stock_df["target_up_7d"].isna()].copy()

print(f"Model-ready rows: {len(stock_df)}")
print(f"Training rows with known 7-day target: {len(training_df)}")
print(f"Latest rows for future prediction only: {len(latest_prediction_df)}")

stock_df[[
    "date", "ticker", "close", "future_close_7d",
    "future_return_7d", "target_up_7d"
]].head(10)


In [ ]:
stock_df.to_csv("data/processed/model_ready_stock_data.csv", index=False)
training_df.to_csv("data/processed/training_ready_stock_data.csv", index=False)
latest_prediction_df.to_csv("data/processed/latest_prediction_rows.csv", index=False)


## 10. First Model

This section builds a first version of the model. The goal is to predict whether a stock is up or not up 7 trading days later.

I used a time-based split so the model trains on earlier dates and tests on later dates. This is better for stock data than randomly mixing old and newer rows.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

model_df = pd.read_csv("data/processed/training_ready_stock_data.csv")
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values(["date", "ticker"])

numeric_features = [
    "daily_return", "return_7d", "return_30d", "ma_7d", "ma_30d",
    "volatility_30d", "volume_change", "avg_sentiment_score", "article_count"
]
feature_columns = numeric_features + ["ticker"]

model_df = model_df.dropna(subset=feature_columns + ["target_up_7d"]).copy()
model_df["target_up_7d"] = model_df["target_up_7d"].astype(int)

split_date = model_df["date"].quantile(0.80)
train_df = model_df[model_df["date"] <= split_date].copy()
test_df = model_df[model_df["date"] > split_date].copy()

X_train = train_df[feature_columns]
y_train = train_df["target_up_7d"]
X_test = test_df[feature_columns]
y_test = test_df["target_up_7d"]

print(f"Training rows: {len(train_df)}")
print(f"Testing rows: {len(test_df)}")
print(f"Split date: {split_date.date()}")
print("Training target balance:")
print(y_train.value_counts(normalize=True).round(3))
print("Testing target balance:")
print(y_test.value_counts(normalize=True).round(3))


In [ ]:
preprocessor_scaled = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("ticker", OneHotEncoder(handle_unknown="ignore"), ["ticker"])
])

preprocessor_plain = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    ("ticker", OneHotEncoder(handle_unknown="ignore"), ["ticker"])
])

models = {
    "Baseline Majority Class": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor_plain),
        ("model", DecisionTreeClassifier(
            max_depth=2,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_plain),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            max_features=None,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

model_results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[name] = model
    model_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "f1_up": f1_score(y_test, predictions),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "split_date": split_date.date()
    })

results_df = pd.DataFrame(model_results).sort_values("balanced_accuracy", ascending=False)
results_df


In [ ]:
model_candidates = results_df[results_df["model"] != "Baseline Majority Class"]
best_model_name = model_candidates.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print(f"Selected model: {best_model_name}")
print("Accuracy:", round(accuracy_score(y_test, best_predictions), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, best_predictions), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))
print("Classification report:")
print(classification_report(y_test, best_predictions))

test_predictions_df = test_df[["date", "ticker", "close", "target_up_7d", "future_return_7d"]].copy()
test_predictions_df["predicted_up_7d"] = best_predictions

if hasattr(best_model, "predict_proba"):
    test_predictions_df["prediction_probability_up"] = best_model.predict_proba(X_test)[:, 1]

results_df.to_csv("outputs/model_results/model_comparison.csv", index=False)
test_predictions_df.to_csv("outputs/model_results/test_set_predictions.csv", index=False)

test_predictions_df.head()


In [ ]:
selected_estimator = best_model.named_steps["model"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
feature_names = [name.replace("numeric__", "").replace("ticker__", "ticker_") for name in feature_names]

if hasattr(selected_estimator, "feature_importances_"):
    importance_values = selected_estimator.feature_importances_
    importance_label = "importance"
elif hasattr(selected_estimator, "coef_"):
    importance_values = abs(selected_estimator.coef_[0])
    importance_label = "absolute_coefficient"
else:
    importance_values = np.zeros(len(feature_names))
    importance_label = "importance"

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    importance_label: importance_values
}).sort_values(importance_label, ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df["feature"], feature_importance_df[importance_label])
plt.title(f"{best_model_name} Feature Importance")
plt.xlabel(importance_label.replace("_", " ").title())
plt.tight_layout()
plt.savefig("outputs/charts/model_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

feature_importance_df.to_csv("outputs/model_results/model_feature_importance.csv", index=False)
feature_importance_df.sort_values(importance_label, ascending=False)


## 11. Current Prediction Rows

After training the model on historical rows, this section applies it to the newest rows where the 7-day result is not known yet. These are prediction rows only, not training rows.


In [ ]:
latest_df = pd.read_csv("data/processed/latest_prediction_rows.csv")
latest_df["date"] = pd.to_datetime(latest_df["date"])
latest_features = latest_df.dropna(subset=feature_columns).copy()

latest_predictions = best_model.predict(latest_features[feature_columns])
latest_output = latest_features[["date", "ticker", "close", "avg_sentiment_score", "article_count"]].copy()
latest_output["predicted_up_7d"] = latest_predictions

if hasattr(best_model, "predict_proba"):
    latest_output["prediction_probability_up"] = best_model.predict_proba(latest_features[feature_columns])[:, 1]

latest_output = latest_output.sort_values(["date", "ticker"])
latest_output.to_csv("outputs/model_results/latest_7d_direction_predictions.csv", index=False)
latest_output.tail(12)


## 12. Next Steps

The next step is to review the model results, decide which model is easiest to explain, and use the predictions/results in the final notebook, dashboard, or presentation.
